In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

cic_features = pd.read_csv("../data/processed/cicids2017_features_reduced.csv").columns.tolist()
unsw_features = pd.read_csv("../data/processed/unsw_train_encoded.csv").columns.tolist()

print("CICIDS2017 features:", len(cic_features))
print("UNSW_NB15 features:", len(unsw_features))
print("\nExact name overlap:", set(cic_features) & set(unsw_features))

CICIDS2017 features: 46
UNSW_NB15 features: 78

Exact name overlap: set()


In [4]:
print("CICIDS2017 columns:")
print(cic_features)

print("\nUNSW-NB15 columns (ecxluding one-hot encoded proto/service/state columns):")
unsw_base_cols = [c for c in unsw_features if not any(c.startswith(p) for p in ['proto_', 'service_', 'state_'])]
print(unsw_base_cols)

CICIDS2017 columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Min', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Bwd Packets/s', 'Min Packet Length', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'Down/Up Ratio', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std']

UNSW-NB15 columns (ecxluding one-hot encoded proto/service/state columns):
['id', 'dur', 's

In [9]:
# Build alligned datatsets using only these 10 common features

feature_map= {
    'dur': 'Flow Duration',
    'spkts': 'Total Fwd Packets',
    'sbytes': 'Total Length of Fwd Packets',
    'smean': 'Fwd Packet Length Mean',
    'dmean': 'Bwd Packet Length Max',
    'rate': 'Flow Packets/s',
    'sinpkt': 'Flow IAT Mean',
    'dinpkt': 'Bwd IAT Mean',
    'swin': 'Init_Win_bytes_forward',
    'dwin': 'Init_Win_bytes_backward'
}

common_cic_cols = list(feature_map.values())

# CICIDS2017
X = pd.read_csv("../data/processed/cicids2017_features_reduced.csv")
y = pd.read_csv("../data/processed/cicids2017_labels.csv")
X_cic_common = X[common_cic_cols]

# UNSW-NB15 train + test
unsw_train_raw = pd.read_csv("../data/raw/unsw-nb15/UNSW_NB15_training-set.csv")
unsw_test_raw = pd.read_csv("../data/raw/unsw-nb15/UNSW_NB15_testing-set.csv")

X_unsw_common = unsw_train_raw[list(feature_map.keys())].rename(columns=feature_map)
y_unsw_common = unsw_train_raw['label']

X_unsw_test_common = unsw_test_raw[list(feature_map.keys())].rename(columns=feature_map)
y_unsw_test_common = unsw_test_raw['label']

print("CICIDS2017 common shape:", X_cic_common.shape)
print("UNSW-NB15 commom shape:", X_unsw_common.shape)


CICIDS2017 common shape: (610492, 10)
UNSW-NB15 commom shape: (175341, 10)


In [10]:
# Train fresh on CICIDS2017, test on UNSW-NB15

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X_train_cd, X_test_cd, y_train_cd, y_test_cd = train_test_split(X_cic_common, y, test_size=0.2, random_state=42, stratify=y)
scaler_cd = StandardScaler()
X_train_cd_scaled = scaler_cd.fit_transform(X_train_cd)
X_test_cd_scaled = scaler_cd.transform(X_test_cd)
X_unsw_scaled = scaler_cd.transform(X_unsw_test_common)

xgb_cd = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_cd.fit(X_train_cd_scaled, y_train_cd)

# Sanity check
y_pred_same = xgb_cd.predict(X_test_cd_scaled)
print("--- CICIDS2017 -> CICIDS2017 (same dataset, 10 fetures) ---")
print("Accuracy:", accuracy_score(y_test_cd, y_pred_same))
print("F1:", f1_score(y_test_cd, y_pred_same))

# Real test
y_pred_cross = xgb_cd.predict(X_unsw_scaled)
print("\n--- CICIDS2017 -> USNW-NB15 (cross dataset) ---")
print("Accuracy:", accuracy_score(y_unsw_test_common, y_pred_cross))
print("Precision:", precision_score(y_unsw_test_common, y_pred_cross))
print("Recall:", recall_score(y_unsw_test_common, y_pred_cross))
print("F1:", f1_score(y_unsw_test_common, y_pred_cross))

--- CICIDS2017 -> CICIDS2017 (same dataset, 10 fetures) ---
Accuracy: 0.9996805870645952
F1: 0.999496975403387

--- CICIDS2017 -> USNW-NB15 (cross dataset) ---
Accuracy: 0.4494121362289268
Precision: 0.6
Recall: 6.617841701226507e-05
F1: 0.00013234223702494652


In [11]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_unsw_test_common, y_pred_cross)
print(cm)
print("\n Predicted class distribution:", pd.Series(y_pred_cross).value_counts().to_dict())
print("Actual class distribution:", y_unsw_test_common.value_counts().to_dict())

[[36998     2]
 [45329     3]]

 Predicted class distribution: {0: 82327, 1: 5}
Actual class distribution: {1: 45332, 0: 37000}


In [12]:
cross_dataset_results = pd.DataFrame([{
    "experiment": "CICIDS2017 -> CICIDS2017 (same dataset, 10 fetures)",
    "accuracy": accuracy_score(y_test_cd, y_pred_same),
    "f1": f1_score(y_test_cd, y_pred_same)
},{
    "experiment": "CICIDS2017 -> USNW-NB15 (cross dataset)",
    "accuracy": accuracy_score(y_unsw_test_common, y_pred_cross),
    "precision": precision_score(y_unsw_test_common, y_pred_cross),
    "recall": recall_score(y_unsw_test_common, y_pred_cross),
    "f1": f1_score(y_unsw_test_common, y_pred_cross)
}])

cross_dataset_results.to_csv("../data/processed/cross_dataset_results.csv", index=False)
print(cross_dataset_results)

                                          experiment  accuracy        f1  \
0  CICIDS2017 -> CICIDS2017 (same dataset, 10 fet...  0.999681  0.999497   
1            CICIDS2017 -> USNW-NB15 (cross dataset)  0.449412  0.000132   

   precision    recall  
0        NaN       NaN  
1        0.6  0.000066  
